# COND-EXP-001 - BDD100K Condition Profile Classifier

Amaç: canlı video frame'inden `condition_profile` üretmek için hafif bir classifier/router baseline kurmak.

Varsayılan model: `MobileNetV3-Small`.

Opsiyonel challenger: `ResNet18`.

Bu notebook önceki BDD100K notebook'tan alınan dersleri uygular:

- Google Drive önce mount edilir.
- Büyük zip arşivleri Drive'da saklanır, küçük dosya extraction/training local `/content` altında yapılır.
- Mevcut arşiv ve metadata varsa tekrar indirme/extract zorlanmaz.
- Boş veya eksik metadata CSV geçerli sayılmaz.
- Notebook restart sonrası Drive'daki `condition_metadata.csv` ve model checkpointlerini yeniden kullanabilir.
- Ham veri, model ağırlığı ve frame çıktıları Git'e eklenmez; Drive/runs altında tutulur.

Önemli ayrım: Bu model araç tespiti yapmaz. Yalnız hava/ışık/görüş profili ve router sinyali üretir.

In [ ]:
# 1) Runtime setup + Drive mount
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', repr(exc))

# Colab'da torch genellikle kurulu gelir; eksik paketleri tamamla.
REQUIRED = [
    'torch',
    'torchvision',
    'pandas',
    'scikit-learn',
    'opencv-python',
    'matplotlib',
    'tqdm',
    'Pillow',
    'tabulate',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *REQUIRED], check=False)

Mounted at /content/drive


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'pandas', 'scikit-learn', 'opencv-python', 'matplotlib', 'tqdm', 'Pillow', 'tabulate'], returncode=0)

In [ ]:
# 2) Config
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
DATASETS_ROOT = PROJECT_ROOT / 'datasets'
BDD_ROOT = DATASETS_ROOT / 'bdd100k'
VEHICLE_YOLO_ROOT = DATASETS_ROOT / 'bdd100k_vehicle_yolo'
COND_ROOT = DATASETS_ROOT / 'bdd100k_condition_classifier'
RUN_ROOT = PROJECT_ROOT / 'runs' / 'condition_profile' / 'COND-EXP-001'
REPORT_ROOT = PROJECT_ROOT / 'runs' / 'condition_profile' / 'COND-EXP-001' / 'reports'
LOCAL_ROOT = Path('/content/anomali-road-safety-ai-condition-work')
LOCAL_ARCHIVE_ROOT = LOCAL_ROOT / 'archives'
LOCAL_BDD_ROOT = LOCAL_ROOT / 'bdd100k'

# Existing Drive archives from prior VD-EXP-002 work. Notebook uses these if present.
BDD_IMAGES_ARCHIVE = BDD_ROOT / 'bdd100k_images.zip'
BDD_LABELS_ARCHIVE = BDD_ROOT / 'bdd100k_det_20_labels_trainval.zip'

# Fallback URLs are only used if the archive is missing and downloads are enabled.
ALLOW_NETWORK_DOWNLOAD = False
BDD_IMAGES_URLS = [
    'https://archive.org/download/bdd100k/bdd100k_images.zip',
]
BDD_LABELS_URLS = [
    'https://dl.cv.ethz.ch/bdd100k/data/bdd100k_det_20_labels_trainval.zip',
    'https://archive.org/download/bdd100k/bdd100k_labels.zip',
]

# Dataset / training controls
REBUILD_METADATA = False
SMOKE_MODE = False
SMOKE_IMAGES_PER_CLASS = 250
MAX_IMAGES_PER_CLASS = 6000
MIN_IMAGES_PER_CLASS = 20
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 96
EPOCHS = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
# Colab Drive + multiprocessing can emit noisy DataLoader cleanup warnings; keep default stable.
NUM_WORKERS = 0
PATIENCE = 3

# Heavy comparison mode: train MobileNetV3-Small and ResNet18 in the same run.
RUN_MOBILENETV3_SMALL = True
RUN_RESNET18 = True
EXPORT_ONNX = False

# Use validation for model selection; keep test metrics for final reporting only.
MODEL_SELECTION_METRIC = 'best_val_macro_f1'

# Optional smoke test after training. Put Test/video_1-3.mp4 under Drive project root if wanted.
RUN_DARK_VIDEO_SMOKE = True
DARK_VIDEO_DIR_CANDIDATES = [
    PROJECT_ROOT / 'Test',
    PROJECT_ROOT / 'test',
    PROJECT_ROOT / 'datasets' / 'test_videos',
]
DARK_VIDEO_DIR = next((path for path in DARK_VIDEO_DIR_CANDIDATES if path.exists()), DARK_VIDEO_DIR_CANDIDATES[0])
DARK_VIDEO_SAMPLE_EVERY_N_FRAMES = 15

# Runtime router policy constants. Specialists are not promoted yet.
CONDITION_CONFIDENCE_THRESHOLD = 0.65
SPECIALISTS_PROVEN_BETTER = {
    'night_low_light': False,
    'rain': False,
    'fog_low_visibility': False,
}

for path in [COND_ROOT, RUN_ROOT, REPORT_ROOT, LOCAL_ROOT, LOCAL_ARCHIVE_ROOT, LOCAL_BDD_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('BDD_ROOT:', BDD_ROOT)
print('COND_ROOT:', COND_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)
print('RUN_MOBILENETV3_SMALL:', RUN_MOBILENETV3_SMALL, 'RUN_RESNET18:', RUN_RESNET18)


PROJECT_ROOT: /content/drive/MyDrive/anomali-road-safety-ai
BDD_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k
COND_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_condition_classifier
RUN_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/runs/condition_profile/COND-EXP-001
LOCAL_ROOT: /content/anomali-road-safety-ai-condition-work
RUN_MOBILENETV3_SMALL: True RUN_RESNET18: True


In [ ]:
# 3) Utilities: archive handling, metadata guards, condition mapping
import csv
import hashlib
import json
import random
import shutil
import subprocess
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REQUIRED_METADATA_COLUMNS = {
    'image_name',
    'image_path',
    'source_split',
    'condition_profile',
    'weather',
    'timeofday',
    'scene',
}

CONDITION_CLASSES = [
    'day_clear',
    'night_low_light',
    'low_light_transition',
    'rain',
    'fog_low_visibility',
    'adverse_other',
    'unknown',
]

LABEL_TO_ID = {label: idx for idx, label in enumerate(CONDITION_CLASSES)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}


def stable_hash(text: str) -> int:
    return int(hashlib.sha1(text.encode('utf-8')).hexdigest()[:12], 16)


def normalize_value(value) -> str:
    if value is None:
        return 'unknown'
    text = str(value).strip().lower().replace(' ', '_').replace('-', '_')
    return text or 'unknown'


def map_condition_profile(attributes: dict) -> str:
    weather = normalize_value(attributes.get('weather'))
    timeofday = normalize_value(attributes.get('timeofday'))
    scene = normalize_value(attributes.get('scene'))

    # Priority order matters: adverse weather should not be hidden by time of day.
    if weather in {'foggy', 'fog', 'mist', 'haze'}:
        return 'fog_low_visibility'
    if weather in {'rainy', 'rain', 'wet'}:
        return 'rain'
    if timeofday in {'night', 'nighttime'}:
        return 'night_low_light'
    if timeofday in {'dawn/dusk', 'dawn_dusk', 'dawn', 'dusk', 'twilight'}:
        return 'low_light_transition'
    if weather in {'snowy', 'snow', 'sandstorm', 'storm', 'undefined'}:
        return 'adverse_other'
    if timeofday in {'daytime', 'day'}:
        return 'day_clear'
    if scene in {'tunnel', 'parking_lot', 'garage'}:
        return 'adverse_other'
    return 'unknown'


def dedupe_condition_metadata_df(df: pd.DataFrame) -> pd.DataFrame:
    dedup_keys = [
        'image_name',
        'source_split',
        'weather',
        'timeofday',
        'scene',
        'condition_profile',
    ]
    existing_keys = [key for key in dedup_keys if key in df.columns]
    if not existing_keys:
        return df
    before = len(df)
    df = df.drop_duplicates(subset=existing_keys).reset_index(drop=True)
    removed = before - len(df)
    if removed:
        print(f'Dropped duplicate condition metadata rows: {removed}')
    return df


def valid_metadata_csv(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        df = pd.read_csv(path, nrows=5)
    except Exception as exc:
        print('Metadata read failed:', path, repr(exc))
        return False
    missing = REQUIRED_METADATA_COLUMNS - set(df.columns)
    if missing:
        print('Metadata missing columns:', sorted(missing))
        return False
    return True


def list_tree_preview(root: Path, max_items: int = 80) -> None:
    print('Tree preview:', root)
    if not root.exists():
        print('  missing')
        return
    count = 0
    for path in root.rglob('*'):
        print(' ', path.relative_to(root))
        count += 1
        if count >= max_items:
            break


def download_file_with_fallback(urls, target_path: Path, artifact_name: str, min_size_mb: float = 1.0) -> Path:
    target_path.parent.mkdir(parents=True, exist_ok=True)
    if target_path.exists() and target_path.stat().st_size >= min_size_mb * 1024 * 1024:
        print(f'{artifact_name} already exists:', target_path)
        return target_path
    if not ALLOW_NETWORK_DOWNLOAD:
        raise FileNotFoundError(
            f'{artifact_name} is missing at {target_path} and ALLOW_NETWORK_DOWNLOAD=False. '
            'Place the archive in Drive or enable downloads.'
        )
    errors = []
    for url in urls:
        clean_url = str(url).strip()
        print('Trying URL:', clean_url)
        try:
            subprocess.run(['wget', '-O', str(target_path) + '.part', '--tries=5', '--timeout=30', clean_url], check=True)
            part = Path(str(target_path) + '.part')
            if part.exists() and part.stat().st_size >= min_size_mb * 1024 * 1024:
                part.rename(target_path)
                return target_path
        except Exception as exc:
            errors.append(repr(exc))
            print('Download failed:', repr(exc))
    raise RuntimeError(f'Could not download {artifact_name}. Errors: {errors}')


def copy_archive_to_local(drive_archive: Path, local_name: str, urls, min_size_mb: float, artifact_name: str) -> Path:
    if not drive_archive.exists() or drive_archive.stat().st_size < min_size_mb * 1024 * 1024:
        print('Drive archive missing or too small:', drive_archive)
        drive_archive = download_file_with_fallback(urls, drive_archive, artifact_name, min_size_mb)
    local_archive = LOCAL_ARCHIVE_ROOT / local_name
    if local_archive.exists() and local_archive.stat().st_size == drive_archive.stat().st_size:
        print('Local archive cache already exists:', local_archive)
        return local_archive
    print('Copying archive to local runtime cache:', drive_archive, '->', local_archive)
    shutil.copy2(drive_archive, local_archive)
    return local_archive


def extract_zip_once(zip_path: Path, extract_root: Path, marker_name: str) -> None:
    marker = extract_root / marker_name
    if marker.exists():
        print('Extraction marker exists, skipping:', marker)
        return
    extract_root.mkdir(parents=True, exist_ok=True)
    print('Extracting:', zip_path, '->', extract_root)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_root)
    marker.write_text(json.dumps({'zip': str(zip_path), 'status': 'extracted'}, indent=2), encoding='utf-8')


def find_image_files(root: Path) -> dict[str, str]:
    image_index = {}
    image_paths = list(root.rglob('*.jpg')) + list(root.rglob('*.jpeg')) + list(root.rglob('*.png'))
    print('Image files found:', len(image_paths))
    for path in tqdm(image_paths, desc='Indexing images'):
        stem = path.stem
        image_index[stem] = str(path)
        image_index[path.name] = str(path)
    return image_index


def infer_split_from_path(path: Path) -> str:
    parts = {part.lower() for part in path.parts}
    if 'train' in parts:
        return 'train'
    if 'val' in parts or 'valid' in parts:
        return 'val'
    if 'test' in parts:
        return 'test'
    return 'unknown'


def iter_label_records_from_json(path: Path):
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except UnicodeDecodeError:
        data = json.loads(path.read_text())

    source_split = infer_split_from_path(path)

    def emit(item, fallback_name=None):
        if not isinstance(item, dict):
            return
        name = item.get('name') or item.get('image') or item.get('image_name') or fallback_name or path.stem
        attrs = item.get('attributes') or item.get('attrs') or {}
        if isinstance(attrs, dict):
            yield {
                'image_name': str(name),
                'source_split': source_split,
                'weather': normalize_value(attrs.get('weather')),
                'timeofday': normalize_value(attrs.get('timeofday')),
                'scene': normalize_value(attrs.get('scene')),
                'condition_profile': map_condition_profile(attrs),
                'label_json': str(path),
            }
        # Some BDD variants keep frame-level entries.
        for frame in item.get('frames', []) if isinstance(item.get('frames'), list) else []:
            frame_name = frame.get('name') or name
            frame_attrs = frame.get('attributes') or attrs
            if isinstance(frame_attrs, dict):
                yield {
                    'image_name': str(frame_name),
                    'source_split': source_split,
                    'weather': normalize_value(frame_attrs.get('weather')),
                    'timeofday': normalize_value(frame_attrs.get('timeofday')),
                    'scene': normalize_value(frame_attrs.get('scene')),
                    'condition_profile': map_condition_profile(frame_attrs),
                    'label_json': str(path),
                }

    if isinstance(data, list):
        for item in data:
            yield from emit(item)
    elif isinstance(data, dict):
        if 'frames' in data or 'attributes' in data or 'name' in data:
            yield from emit(data)
        for key in ['labels', 'items', 'images', 'annotations']:
            if isinstance(data.get(key), list):
                for item in data[key]:
                    yield from emit(item)


def build_condition_metadata(label_root: Path, image_root: Path, output_csv: Path) -> pd.DataFrame:
    print('Building image index from:', image_root)
    image_index = find_image_files(image_root)
    json_files = sorted(label_root.rglob('*.json'))
    print('Label JSON files found:', len(json_files))
    if not json_files:
        list_tree_preview(label_root)
        raise FileNotFoundError('No label JSON files found under local label root.')

    rows = []
    missing_images = 0
    for path in tqdm(json_files, desc='Reading label JSON'):
        for rec in iter_label_records_from_json(path):
            name = rec['image_name']
            image_path = image_index.get(Path(name).stem) or image_index.get(name)
            if not image_path:
                missing_images += 1
                continue
            rec['image_path'] = image_path
            rows.append(rec)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'Condition metadata produced 0 rows. Missing image matches: {missing_images}')
    df = dedupe_condition_metadata_df(df)
    missing = REQUIRED_METADATA_COLUMNS - set(df.columns)
    if missing:
        raise RuntimeError('Condition metadata missing columns: ' + ', '.join(sorted(missing)))
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)
    print('Wrote condition metadata:', output_csv, df.shape)
    print('Missing image matches skipped:', missing_images)
    return df


In [ ]:
# 4) Prepare local BDD100K labels/images and build condition metadata
metadata_csv = COND_ROOT / 'metadata' / 'condition_metadata.csv'
metadata_csv.parent.mkdir(parents=True, exist_ok=True)


def ensure_local_bdd_archives_extracted() -> None:
    labels_zip = copy_archive_to_local(
        BDD_LABELS_ARCHIVE,
        'bdd100k_labels.zip',
        BDD_LABELS_URLS,
        min_size_mb=10,
        artifact_name='BDD100K labels archive',
    )
    images_zip = copy_archive_to_local(
        BDD_IMAGES_ARCHIVE,
        'bdd100k_images.zip',
        BDD_IMAGES_URLS,
        min_size_mb=100,
        artifact_name='BDD100K images archive',
    )
    extract_zip_once(labels_zip, LOCAL_BDD_ROOT, '.labels_extracted.marker')
    extract_zip_once(images_zip, LOCAL_BDD_ROOT, '.images_extracted.marker')


def sample_image_paths_exist(df: pd.DataFrame, sample_size: int = 100) -> float:
    if df.empty or 'image_path' not in df.columns:
        return 0.0
    sample = df['image_path'].dropna().astype(str).head(sample_size).tolist()
    if not sample:
        return 0.0
    return sum(Path(path).exists() for path in sample) / len(sample)

if valid_metadata_csv(metadata_csv) and not REBUILD_METADATA:
    metadata_df = pd.read_csv(metadata_csv)
    print('Found existing condition metadata:', metadata_csv, metadata_df.shape)
    exists_ratio = sample_image_paths_exist(metadata_df)
    print('Existing metadata local image path sample exists ratio:', exists_ratio)
    if exists_ratio < 0.80:
        print('Local /content image paths are missing or stale. Re-extracting archives into the same local workdir.')
        ensure_local_bdd_archives_extracted()
        exists_ratio = sample_image_paths_exist(metadata_df)
        print('Post-extract image path exists ratio:', exists_ratio)
    if exists_ratio < 0.80:
        print('Existing metadata still stale after extraction. Rebuilding metadata from local labels/images.')
        metadata_df = build_condition_metadata(LOCAL_BDD_ROOT, LOCAL_BDD_ROOT, metadata_csv)
else:
    ensure_local_bdd_archives_extracted()
    list_tree_preview(LOCAL_BDD_ROOT, max_items=80)
    metadata_df = build_condition_metadata(LOCAL_BDD_ROOT, LOCAL_BDD_ROOT, metadata_csv)

if sample_image_paths_exist(metadata_df) < 0.80:
    raise RuntimeError('Condition metadata is valid structurally, but image paths are not readable in this runtime.')

before_dedupe = len(metadata_df)
metadata_df = dedupe_condition_metadata_df(metadata_df)
if len(metadata_df) != before_dedupe:
    metadata_df.to_csv(metadata_csv, index=False)
    print('Persisted deduplicated condition metadata:', metadata_csv, metadata_df.shape)

print('Metadata rows:', metadata_df.shape)
print(metadata_df['condition_profile'].value_counts(dropna=False))
display(metadata_df.head())


Found existing condition metadata: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_condition_classifier/metadata/condition_metadata.csv (160000, 8)
Existing metadata local image path sample exists ratio: 0.0
Local /content image paths are missing or stale. Re-extracting archives into the same local workdir.
Copying archive to local runtime cache: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k/bdd100k_det_20_labels_trainval.zip -> /content/anomali-road-safety-ai-condition-work/archives/bdd100k_labels.zip
Copying archive to local runtime cache: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k/bdd100k_images.zip -> /content/anomali-road-safety-ai-condition-work/archives/bdd100k_images.zip
Extracting: /content/anomali-road-safety-ai-condition-work/archives/bdd100k_labels.zip -> /content/anomali-road-safety-ai-condition-work/bdd100k
Extracting: /content/anomali-road-safety-ai-condition-work/archives/bdd100k_images.zip -> /content/anomali-road-saf

,image_name,source_split,weather,timeofday,scene,condition_profile,label_json,image_path
0,0000f77c-6257be58,train,clear,daytime,city_street,day_clear,/content/anomali-road-safety-ai-condition-work...,/content/anomali-road-safety-ai-condition-work...
1,0000f77c-62c2a288,train,clear,dawn/dusk,highway,low_light_transition,/content/anomali-road-safety-ai-condition-work...,/content/anomali-road-safety-ai-condition-work...
2,0000f77c-cb820c98,train,clear,dawn/dusk,residential,low_light_transition,/content/anomali-road-safety-ai-condition-work...,/content/anomali-road-safety-ai-condition-work...
3,0001542f-5ce3cf52,train,clear,night,city_street,night_low_light,/content/anomali-road-safety-ai-condition-work...,/content/anomali-road-safety-ai-condition-work...
4,0001542f-7c670be8,train,clear,night,highway,night_low_light,/content/anomali-road-safety-ai-condition-work...,/content/anomali-road-safety-ai-condition-work...


In [ ]:
# 5) Build balanced train/val/test split CSVs
split_root = COND_ROOT / 'splits'
split_root.mkdir(parents=True, exist_ok=True)

work_df = metadata_df.copy()
work_df = work_df[work_df['condition_profile'].isin(CONDITION_CLASSES)].copy()
work_df['label_id'] = work_df['condition_profile'].map(LABEL_TO_ID)
work_df['hash'] = work_df['image_name'].astype(str).map(stable_hash)

# Remove extremely small classes unless you intentionally keep them.
counts = work_df['condition_profile'].value_counts()
keep_classes = [cls for cls, count in counts.items() if count >= MIN_IMAGES_PER_CLASS]
work_df = work_df[work_df['condition_profile'].isin(keep_classes)].copy()
print('Kept classes:', keep_classes)
print('Raw kept distribution:')
print(work_df['condition_profile'].value_counts())

# Balanced cap per class to keep first Colab run practical.
limit_per_class = SMOKE_IMAGES_PER_CLASS if SMOKE_MODE else MAX_IMAGES_PER_CLASS
balanced_parts = []
for cls, part in work_df.groupby('condition_profile'):
    part = part.sort_values('hash')
    if limit_per_class:
        part = part.head(limit_per_class)
    balanced_parts.append(part)
balanced_df = pd.concat(balanced_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# Use official train as train. Split official val into val/test. If no val exists, hash-split all.
def assign_split(row):
    source = str(row.get('source_split', 'unknown')).lower()
    bucket = int(row['hash'] % 100)
    if source == 'train':
        return 'train'
    if source in {'val', 'valid'}:
        return 'val' if bucket < 50 else 'test'
    if bucket < 70:
        return 'train'
    if bucket < 85:
        return 'val'
    return 'test'

balanced_df['split'] = balanced_df.apply(assign_split, axis=1)

# Guard against missing val/test for small classes.
for cls in sorted(balanced_df['condition_profile'].unique()):
    cls_idx = balanced_df.index[balanced_df['condition_profile'] == cls].tolist()
    splits = set(balanced_df.loc[cls_idx, 'split'])
    if 'val' not in splits and len(cls_idx) >= 3:
        balanced_df.loc[cls_idx[0], 'split'] = 'val'
    if 'test' not in splits and len(cls_idx) >= 4:
        balanced_df.loc[cls_idx[1], 'split'] = 'test'

for split in ['train', 'val', 'test']:
    out = split_root / f'{split}.csv'
    split_df = balanced_df[balanced_df['split'] == split].copy()
    split_df.to_csv(out, index=False)
    print(split, split_df.shape, '->', out)

class_distribution = (
    balanced_df.groupby(['split', 'condition_profile'])
    .size()
    .reset_index(name='count')
    .sort_values(['split', 'condition_profile'])
)
class_distribution.to_csv(COND_ROOT / 'metadata' / 'class_distribution.csv', index=False)
print('Distribution:')
display(class_distribution)

label_map_path = COND_ROOT / 'metadata' / 'condition_label_map.json'
label_map_path.write_text(json.dumps({'label_to_id': LABEL_TO_ID, 'id_to_label': ID_TO_LABEL}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Label map:', label_map_path)

Kept classes: ['night_low_light', 'day_clear', 'adverse_other', 'rain', 'low_light_transition', 'fog_low_visibility']
Raw kept distribution:
condition_profile
night_low_light         29388
day_clear               27747
adverse_other           11488
rain                     5821
low_light_transition     5411
fog_low_visibility        143
Name: count, dtype: int64
train (25636, 11) -> /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_condition_classifier/splits/train.csv
val (1859, 11) -> /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_condition_classifier/splits/val.csv
test (1880, 11) -> /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_condition_classifier/splits/test.csv
Distribution:


,split,condition_profile,count
0,test,adverse_other,385
1,test,day_clear,412
2,test,fog_low_visibility,5
3,test,low_light_transition,364
4,test,night_low_light,351
5,test,rain,363
6,train,adverse_other,5247
7,train,day_clear,5217
8,train,fog_low_visibility,130
9,train,low_light_transition,4689


Label map: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_condition_classifier/metadata/condition_label_map.json


In [ ]:
# 6) Training helpers
import math
import time
from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.models import MobileNet_V3_Small_Weights, ResNet18_Weights

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

class ConditionFrameDataset(Dataset):
    def __init__(self, csv_path: Path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.df = self.df[self.df['image_path'].map(lambda p: Path(str(p)).exists())].reset_index(drop=True)
        self.transform = transform
        if self.df.empty:
            raise RuntimeError(f'No readable images in {csv_path}')
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = int(row['label_id'])
        return image, label

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.10, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ConditionFrameDataset(split_root / 'train.csv', transform=train_tf)
val_ds = ConditionFrameDataset(split_root / 'val.csv', transform=val_tf)
test_ds = ConditionFrameDataset(split_root / 'test.csv', transform=val_tf)

train_labels = train_ds.df['label_id'].astype(int).tolist()
class_counts = Counter(train_labels)
weights_per_class = {cls: 1.0 / max(count, 1) for cls, count in class_counts.items()}
sample_weights = [weights_per_class[label] for label in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print('Dataset sizes:', len(train_ds), len(val_ds), len(test_ds))
print('Train class counts:', class_counts)


def build_model(backbone: str, num_classes: int):
    if backbone == 'mobilenet_v3_small':
        weights = MobileNet_V3_Small_Weights.DEFAULT
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    if backbone == 'resnet18':
        weights = ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model
    raise ValueError(f'Unknown backbone: {backbone}')


def run_eval(model, loader):
    model.eval()
    losses = []
    y_true, y_pred, y_prob = [], [], []
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            losses.append(float(loss.item()))
            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())
            y_prob.extend(probs.max(dim=1).values.cpu().numpy().tolist())
    return {
        'loss': float(np.mean(losses)) if losses else None,
        'accuracy': float(accuracy_score(y_true, y_pred)) if y_true else None,
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)) if y_true else None,
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)) if y_true else None,
        'mean_confidence': float(np.mean(y_prob)) if y_prob else None,
        'y_true': y_true,
        'y_pred': y_pred,
    }


def train_one_backbone(backbone: str):
    model_dir = RUN_ROOT / 'train' / f'COND-EXP-001-{backbone}'
    model_dir.mkdir(parents=True, exist_ok=True)
    model = build_model(backbone, len(CONDITION_CLASSES)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best_macro_f1 = -1.0
    best_epoch = -1
    no_improve = 0
    history = []
    best_path = model_dir / 'best.pt'

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_losses = []
        start = time.perf_counter()
        for images, labels in tqdm(train_loader, desc=f'{backbone} epoch {epoch}/{EPOCHS}'):
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_losses.append(float(loss.item()))
        val_metrics = run_eval(model, val_loader)
        row = {
            'epoch': epoch,
            'train_loss': float(np.mean(epoch_losses)) if epoch_losses else None,
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
            'val_weighted_f1': val_metrics['weighted_f1'],
            'seconds': round(time.perf_counter() - start, 2),
        }
        history.append(row)
        print(row)
        if val_metrics['macro_f1'] is not None and val_metrics['macro_f1'] > best_macro_f1:
            best_macro_f1 = val_metrics['macro_f1']
            best_epoch = epoch
            no_improve = 0
            torch.save({
                'backbone': backbone,
                'state_dict': model.state_dict(),
                'condition_classes': CONDITION_CLASSES,
                'label_to_id': LABEL_TO_ID,
                'image_size': IMAGE_SIZE,
                'epoch': epoch,
                'val_metrics': {k: v for k, v in val_metrics.items() if not k.startswith('y_')},
            }, best_path)
            print('Saved best checkpoint:', best_path)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print('Early stopping at epoch', epoch)
                break

    checkpoint = torch.load(best_path, map_location=device)
    model = build_model(backbone, len(CONDITION_CLASSES)).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    test_metrics = run_eval(model, test_loader)
    report = classification_report(
        test_metrics['y_true'],
        test_metrics['y_pred'],
        labels=list(range(len(CONDITION_CLASSES))),
        target_names=CONDITION_CLASSES,
        zero_division=0,
        output_dict=True,
    )
    cm = confusion_matrix(test_metrics['y_true'], test_metrics['y_pred'], labels=list(range(len(CONDITION_CLASSES))))
    summary = {
        'experiment_id': 'COND-EXP-001',
        'backbone': backbone,
        'best_epoch': best_epoch,
        'best_val_macro_f1': best_macro_f1,
        'test_metrics': {k: v for k, v in test_metrics.items() if not k.startswith('y_')},
        'classification_report': report,
        'confusion_matrix': cm.tolist(),
        'condition_classes': CONDITION_CLASSES,
        'checkpoint': str(best_path),
        'history': history,
    }
    (model_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    pd.DataFrame(history).to_csv(model_dir / 'history.csv', index=False)
    print('Test metrics:', summary['test_metrics'])
    return summary

device: cuda
Dataset sizes: 25636 1859 1880
Train class counts: Counter({1: 5270, 5: 5247, 0: 5217, 3: 5083, 2: 4689, 4: 130})


In [ ]:
# 7) Train selected backbones
summaries = []
if RUN_MOBILENETV3_SMALL:
    summaries.append(train_one_backbone('mobilenet_v3_small'))
if RUN_RESNET18:
    summaries.append(train_one_backbone('resnet18'))

if not summaries:
    raise RuntimeError('No backbone selected. Enable RUN_MOBILENETV3_SMALL or RUN_RESNET18.')

summary_root = RUN_ROOT / 'summaries'
summary_root.mkdir(parents=True, exist_ok=True)
comparison = pd.DataFrame([
    {
        'backbone': item['backbone'],
        'best_epoch': item['best_epoch'],
        'best_val_macro_f1': item['best_val_macro_f1'],
        'test_accuracy': item['test_metrics']['accuracy'],
        'test_macro_f1': item['test_metrics']['macro_f1'],
        'test_weighted_f1': item['test_metrics']['weighted_f1'],
        'checkpoint': item['checkpoint'],
    }
    for item in summaries
]).sort_values(MODEL_SELECTION_METRIC, ascending=False)
comparison['selection_metric'] = MODEL_SELECTION_METRIC
comparison_path = summary_root / 'backbone_comparison.csv'
comparison.to_csv(comparison_path, index=False)
print('Backbone comparison:', comparison_path)
print('Model selection metric:', MODEL_SELECTION_METRIC)
display(comparison)

best_summary = max(summaries, key=lambda item: item.get(MODEL_SELECTION_METRIC) or -1)
print('Selected best backbone:', best_summary['backbone'], best_summary['checkpoint'])
print('Selected by:', MODEL_SELECTION_METRIC, best_summary.get(MODEL_SELECTION_METRIC))


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 100MB/s] 
/tmp/ipykernel_394/2758784485.py:124: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


mobilenet_v3_small epoch 1/8:   0%|          | 0/268 [00:00<?, ?it/s]

/tmp/ipykernel_394/2758784485.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


{'epoch': 1, 'train_loss': 0.7158909516770449, 'val_loss': 0.700204285979271, 'val_accuracy': 0.7461000537923614, 'val_macro_f1': 0.6201617636714184, 'val_weighted_f1': 0.7419787323072965, 'seconds': 432.7}
Saved best checkpoint: /content/drive/MyDrive/anomali-road-safety-ai/runs/condition_profile/COND-EXP-001/train/COND-EXP-001-mobilenet_v3_small/best.pt


...

resnet18 epoch 6/8:   0%|          | 0/268 [00:00<?, ?it/s]

/tmp/ipykernel_394/2758784485.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


{'epoch': 6, 'train_loss': 0.3213833842370937, 'val_loss': 0.8280008018016816, 'val_accuracy': 0.7525551371705218, 'val_macro_f1': 0.6243426076226309, 'val_weighted_f1': 0.7468768984877467, 'seconds': 414.74}
Early stopping at epoch 6
Test metrics: {'loss': 0.7386181443929672, 'accuracy': 0.7558510638297873, 'macro_f1': 0.6600201728777172, 'weighted_f1': 0.7543666493900315, 'mean_confidence': 0.8336109530894046}
Backbone comparison: /content/drive/MyDrive/anomali-road-safety-ai/runs/condition_profile/COND-EXP-001/summaries/backbone_comparison.csv
Model selection metric: best_val_macro_f1


,backbone,best_epoch,best_val_macro_f1,test_accuracy,test_macro_f1,test_weighted_f1,checkpoint,selection_metric
0,mobilenet_v3_small,3,0.682300,0.764362,0.638826,0.762459,/content/drive/MyDrive/anomali-road-safety-ai/...,best_val_macro_f1
1,resnet18,3,0.666782,0.755851,0.660020,0.754367,/content/drive/MyDrive/anomali-road-safety-ai/...,best_val_macro_f1


Selected best backbone: mobilenet_v3_small /content/drive/MyDrive/anomali-road-safety-ai/runs/condition_profile/COND-EXP-001/train/COND-EXP-001-mobilenet_v3_small/best.pt
Selected by: best_val_macro_f1 0.6823001276953425


In [ ]:
# 8) Optional ONNX export
if EXPORT_ONNX:
    backbone = best_summary['backbone']
    checkpoint = torch.load(best_summary['checkpoint'], map_location=device)
    model = build_model(backbone, len(CONDITION_CLASSES)).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    onnx_path = Path(best_summary['checkpoint']).with_suffix('.onnx')
    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=['image'],
        output_names=['logits'],
        dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=17,
    )
    print('Exported ONNX:', onnx_path)
else:
    print('EXPORT_ONNX=False, skipping ONNX export.')

EXPORT_ONNX=False, skipping ONNX export.


In [ ]:
# 9) Optional dark video smoke test + router decision
from collections import Counter

infer_tf = val_tf

def load_best_model(summary):
    ckpt = torch.load(summary['checkpoint'], map_location=device)
    model = build_model(summary['backbone'], len(CONDITION_CLASSES)).to(device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    return model


def classify_frame(model, frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb)
    tensor = infer_tf(image).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze(0).cpu().numpy()
    idx = int(probs.argmax())
    return ID_TO_LABEL[idx], float(probs[idx]), probs.tolist()


def router_decision(condition_profile: str, confidence: float):
    if confidence < CONDITION_CONFIDENCE_THRESHOLD:
        return {
            'selected_detector_profile': 'general',
            'fallback_used': True,
            'routing_reason': 'condition confidence below threshold',
        }
    if not SPECIALISTS_PROVEN_BETTER.get(condition_profile, False):
        return {
            'selected_detector_profile': 'general',
            'fallback_used': True,
            'routing_reason': f'{condition_profile} specialist is not promoted; general detector remains active',
        }
    return {
        'selected_detector_profile': condition_profile,
        'fallback_used': False,
        'routing_reason': 'specialist promoted and confidence threshold passed',
    }

video_results = []
if RUN_DARK_VIDEO_SMOKE and DARK_VIDEO_DIR.exists():
    model = load_best_model(best_summary)
    videos = sorted(DARK_VIDEO_DIR.glob('video_*.mp4'))
    print('Dark smoke videos found:', videos)
    if not videos:
        print('DARK_VIDEO_DIR exists, but no video_*.mp4 files were found:', DARK_VIDEO_DIR)
    for video in videos:
        cap = cv2.VideoCapture(str(video))
        frame_idx = 0
        preds = []
        confs = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame_idx += 1
            if frame_idx % DARK_VIDEO_SAMPLE_EVERY_N_FRAMES != 0:
                continue
            label, conf, probs = classify_frame(model, frame)
            preds.append(label)
            confs.append(conf)
        cap.release()
        counts = Counter(preds)
        dominant = counts.most_common(1)[0][0] if counts else 'unknown'
        dominant_conf = float(np.mean([c for p, c in zip(preds, confs) if p == dominant])) if counts else 0.0
        row = {
            'video': str(video),
            'sampled_frames': len(preds),
            'profile_counts': dict(counts),
            'dominant_profile': dominant,
            'dominant_confidence_mean': dominant_conf,
            'unknown_ratio': counts.get('unknown', 0) / max(len(preds), 1),
            **router_decision(dominant, dominant_conf),
        }
        video_results.append(row)
        print(row)
elif RUN_DARK_VIDEO_SMOKE:
    print('DARK_VIDEO_DIR not found, skipping dark video smoke test:', DARK_VIDEO_DIR)
    print('Checked candidates:', [str(path) for path in DARK_VIDEO_DIR_CANDIDATES])
else:
    print('RUN_DARK_VIDEO_SMOKE=False')

smoke_path = summary_root / 'dark_video_condition_smoke.json'
smoke_path.write_text(json.dumps(video_results, ensure_ascii=False, indent=2), encoding='utf-8')
print('Wrote smoke summary:', smoke_path)


DARK_VIDEO_DIR not found, skipping dark video smoke test: /content/drive/MyDrive/anomali-road-safety-ai/Test
Checked candidates: ['/content/drive/MyDrive/anomali-road-safety-ai/Test', '/content/drive/MyDrive/anomali-road-safety-ai/test', '/content/drive/MyDrive/anomali-road-safety-ai/datasets/test_videos']
Wrote smoke summary: /content/drive/MyDrive/anomali-road-safety-ai/runs/condition_profile/COND-EXP-001/summaries/dark_video_condition_smoke.json


In [ ]:
# 10) Write report-ready Markdown summary
md_path = REPORT_ROOT / 'cond_exp_001_condition_classifier_summary.md'
lines = [
    '# COND-EXP-001 Condition Profile Classifier Summary',
    '',
    '## Karar',
    '',
    'İlk mobil/edge uyumlu baseline `MobileNetV3-Small` olarak tasarlanmıştır. `ResNet18` opsiyonel challenger olarak aynı notebook içinde çalıştırılabilir.',
    '',
    '## Dataset',
    '',
    f'* Metadata CSV: `{metadata_csv}`',
    f'* Split root: `{split_root}`',
    f'* Class list: `{CONDITION_CLASSES}`',
    '',
    '## Backbone Comparison',
    '',
    f'* Selection metric: `{MODEL_SELECTION_METRIC}`',
    '* Test metrics are reported after validation-based model selection; test set is not used as the selection criterion.',
    '',
]
lines.append(comparison.to_markdown(index=False))
lines.extend([
    '',
    '## Router Policy',
    '',
    f'* Confidence threshold: `{CONDITION_CONFIDENCE_THRESHOLD}`',
    '* Specialist detectorlar VD-EXP-002 sonucunda henüz promoted değildir; bu nedenle router ilk fazda güvenli `general` fallback kullanır.',
    '',
    '## Dark Video Smoke Test',
    '',
])
if video_results:
    lines.append(pd.DataFrame(video_results).to_markdown(index=False))
else:
    lines.append('Dark video smoke test atlandı veya video bulunamadı.')
lines.extend([
    '',
    '## Teknik Sınırlar',
    '',
    '* Bu koşu bir condition-router baseline koşusudur; nihai saha doğruluğu iddiası değildir.',
    '* `fog_low_visibility` sınıfı BDD100K içinde zayıf destekleniyorsa bu sınıf için ACDC/DAWN gibi harici validation gerekir.',
    '* Dark video smoke test boşsa Drive altında `Test/video_1.mp4` - `video_3.mp4` mevcut değildir veya adlandırma uyuşmamıştır.',
    '',
    '## Rapor Notu',
    '',
    'Bu model araç tespiti yapmaz. Yalnız canlı frame için ortam/ışık/hava/görüş profili üretir ve detector router için sinyal sağlar.',
    'Final saha doğruluğu iddiası için harici adverse-condition validation ve manuel video review gereklidir.',
])
md_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('Wrote Markdown summary:', md_path)


Wrote Markdown summary: /content/drive/MyDrive/anomali-road-safety-ai/runs/condition_profile/COND-EXP-001/reports/cond_exp_001_condition_classifier_summary.md


## Beklenen sonraki adım

1. Ağır comparison run için default ayarları koru: `RUN_MOBILENETV3_SMALL=True`, `RUN_RESNET18=True`.
2. Notebook iki backbone'u sırayla eğitir, her biri için kendi `best.pt` checkpoint'ini yazar ve en iyi modeli `best_val_macro_f1` ile seçer.
3. Drive altında `Test/video_1.mp4` - `video_3.mp4` varsa dark video smoke test seçilen modelle çalışır.
4. Eğer `fog_low_visibility` veya `rain` sınıfları yetersiz kalırsa BDD100K dışı ACDC/DAWN/ExDark kaynakları ayrı fazda eklenir.
5. Classifier çıktısı runtime'da specialist detector'ı otomatik aktiflememeli; specialist ancak VD benchmarkında general detector'dan iyi ise `proven_better=true` yapılmalı.
